# PneumoType Agent: an AI triage agent that coordinates our models
A small open language model (**Qwen2.5-1.5B-Instruct**, not trained by us) acts as the **coordinator**. It never sees the X-ray and never diagnoses on its own. It can only **call PneumoType's tools**: the image-quality check, our two CNNs trained from scratch, a second look, a Grad-CAM attention check, the conformal 95% set and the federated all-ages model. It decides which tools to call, reads the results, and writes a triage for the clinician.

**Guardrails:** required safety tools are enforced, and a fixed safety policy overrides the agent if it picks a less urgent triage than the evidence allows. Every override is logged.

**Setup:** Add Input → *Your Work* → **PneumoType** (notebook output), plus *Chest X-Ray Images (Pneumonia)*. **GPU T4** on, Internet on.
*Prototype only. Not medical advice.*

In [ ]:
!pip install -q gradio

In [ ]:
import os, glob, time, io, urllib.request
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE, K = 224, 3
CLASSES = ["normal", "bacterial", "viral"]
FINAL_BONUS, TEMP, UNSURE_CUTOFF = 0.58, 0.95, 0.7     # from the main notebook (Version 4, chosen on validation)
TRACK = {0.5: 0.826, 0.6: 0.867, 0.7: 0.910, 0.8: 0.947, 0.9: 0.977}   # accuracy on test X-rays at/above each confidence
CONF_MIN = 1 - 0.8222   # split conformal (upgrades notebook): label is in the 95% set if its calibrated probability >= this
CHILD_TH, ADULT_TH = 0.42, 0.5   # federated pneumonia threshold per hospital (upgrades notebook)

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found: add the PneumoType notebook output as an input"
    return hits[0]

def block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                         nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True), nn.MaxPool2d(2))
class OurCNN(nn.Module):
    def __init__(self):
        super().__init__(); ch = [1, 32, 64, 128, 256, 512]
        self.features = nn.Sequential(*[block(a, b) for a, b in zip(ch[:-1], ch[1:])])
class PneumoType(nn.Module):
    def __init__(self, n_out=K):
        super().__init__(); self.backbone = OurCNN(); self.drop = nn.Dropout(0.3); self.head = nn.Linear(512, n_out)
    def forward(self, x):
        f = self.backbone.features((x - 0.5) / 0.25)
        return self.head(self.drop(F.adaptive_avg_pool2d(F.relu(f), 1).flatten(1)))

def load(path, n_out):
    m = PneumoType(n_out).to(device); m.load_state_dict(torch.load(path, map_location=device)); return m.eval()
model, model2 = load(find("best.pt"), 3), load(find("best_robust.pt"), 3)
FED_OK = len(glob.glob("/kaggle/input/**/fed_model.pt", recursive=True)) > 0
fed_model = load(find("fed_model.pt"), 2) if FED_OK else None
print("loaded CNN 1, CNN 2", "+ federated model" if FED_OK else "")

eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor()])
tta_tf = T.Compose([T.Resize((248, 248)), T.CenterCrop(IMG_SIZE), T.ToTensor()])

In [ ]:
@torch.no_grad()
def probs_of(m, pil, tf=eval_tf):
    return F.softmax(m(tf(pil.convert("L")).unsqueeze(0).to(device)).float(), 1)[0].cpu().numpy()

def final_probs(pil):
    ps = [probs_of(m, pil, tf) for m in (model, model2) for tf in (eval_tf, tta_tf)]
    p = np.mean(ps, 0); z = np.log(np.clip(p, 1e-8, 1)) / TEMP; z = np.exp(z - z.max()); return z / z.sum()

def gradcam(pil, m, target):
    x = eval_tf(pil.convert("L")).unsqueeze(0).to(device); acts = {}
    def hook(mod, i, o): acts["a"] = o; o.register_hook(lambda g: acts.__setitem__("g", g))
    h = m.backbone.features.register_forward_hook(hook); logits = m(x); h.remove()
    m.zero_grad(); logits[0, target].backward()
    cam = F.relu((acts["g"].mean(dim=(2, 3), keepdim=True) * acts["a"]).sum(1)).squeeze().detach().cpu().numpy()
    cam = np.array(Image.fromarray(cam).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

def input_checks(pil, lung):
    rgb = np.asarray(pil.convert("RGB")).astype(float); g = np.asarray(pil.convert("L")).astype(float)
    colour = float(np.abs(rgb - rgb.mean(2, keepdims=True)).mean()); w, h = pil.size
    out = []
    if colour > 8: out.append(("FAIL", "Image has colour: this may not be a chest X-ray. Result not reliable."))
    else: out.append(("PASS", "Greyscale image, consistent with a radiograph"))
    if min(w, h) < 256: out.append(("WARN", f"Low resolution ({w}x{h}); robust CNN weighted in, but treat with caution"))
    else: out.append(("PASS", f"Resolution OK ({w}x{h})"))
    if g.std() < 35: out.append(("WARN", f"Low contrast (pixel SD {g.std():.0f}); closer to the 'different clinic' images we stress-tested"))
    else: out.append(("PASS", "Contrast OK"))
    if lung < 0.5: out.append(("WARN", f"Only {lung:.0%} of the model's attention is inside the lung region: possible shortcut, needs a human read"))
    else: out.append(("PASS", f"{lung:.0%} of the model's attention is inside the lung region"))
    return out

def make_report(cls, cset, set_txt, probs, conf, agree, unsure, checks, every, fed_child, fed_adult, ms):
    failed = any(c[0] == "FAIL" for c in checks)
    if failed or unsure or len(cset) == 3:
        level, why = "REFER", "the system is not confident enough to triage this X-ray"
    elif cset == ["bacterial"] or (cls == 1 and "bacterial" in cset):
        level, why = "URGENT", "findings suggest bacterial pneumonia"
    elif set(cset) == {"bacterial", "viral"} or (len(cset) == 2 and "normal" not in cset):
        level, why = "REVIEW TODAY", "pneumonia likely; bacterial vs viral cannot be separated from the X-ray alone"
    elif cset == ["viral"]:
        level, why = "REVIEW", "findings suggest viral pneumonia"
    elif cset == ["normal"]:
        level, why = "ROUTINE", "no radiographic pneumonia detected"
    else:
        level, why = "REVIEW", "normal and pneumonia both plausible"
    steps = {
        "URGENT": ["Clinician review today", "Bacterial pneumonia is likely: decide on antibiotics per local guidelines", "Check oxygen saturation and signs of severe illness"],
        "REVIEW TODAY": ["Clinician review today", "Bloodwork such as CRP or procalcitonin can help decide whether antibiotics are needed", "Re-assess if breathing worsens"],
        "REVIEW": ["Clinician review", "Viral pattern: supportive care is usual; safety-net advice for worsening symptoms", "Consider bacterial co-infection if the child is very unwell"],
        "ROUTINE": ["No pneumonia seen on this X-ray", "Clinical judgement still applies: early pneumonia can be invisible on X-ray"],
        "REFER": ["Do not rely on this result", "Send for radiologist read"],
    }[level]
    icon = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌"}
    lines = [f"## Triage: **{level}**", f"*Reason:* {why}.", "",
             "**Findings**",
             f"- Conformal 95% answer: **{set_txt}**",
             f"- Most likely: {CLASSES[int(probs.argmax())]} ({conf:.0%}); bacterial {probs[1]:.0%} · viral {probs[2]:.0%} · normal {probs[0]:.0%}",
             f"- Two independent networks agree: {'yes' if agree else 'no'}"]
    if every: lines.append(f"- Federated all-ages check: pneumonia {every['pneumonia']:.0%} (children's-hospital threshold: {fed_child}; adult: {fed_adult})")
    lines += ["", "**Suggested next steps**"] + [f"{i+1}. {t}" for i, t in enumerate(steps)]
    lines += ["", "**Automatic safety checks**"] + [f"- {icon[a]} {b}" for a, b in checks]
    lines += ["", f"*Generated in {ms:.0f} ms by PneumoType. Prototype decision support, not medical advice; a clinician makes the decision.*"]
    md = "\n".join(lines)
    path = "/kaggle/working/pneumotype_report.md" if os.path.isdir("/kaggle/working") else "pneumotype_report.md"
    open(path, "w").write("# PneumoType triage report\n\n" + md.replace("## ", "### "))
    return md, path

def analyze(img):
    if img is None: return "### Upload or pick an X-ray", None, None, None, None, "", None
    t0 = time.time(); pil = Image.fromarray(img) if not isinstance(img, Image.Image) else img
    log = []; T0 = time.time()
    def act(tool, result): log.append((f"{(time.time() - T0) * 1000:5.0f} ms", tool, result))
    act("plan", "goal: safe triage of one chest X-ray; start with two independent CNNs")
    p1, p2 = probs_of(model, pil), probs_of(model2, pil)
    act("run CNN 1 (original)", f"{CLASSES[int(p1.argmax())]} {p1.max():.0%}")
    act("run CNN 2 (clinic-robust)", f"{CLASSES[int(p2.argmax())]} {p2.max():.0%}")
    probs = final_probs(pil)
    act("ensemble + test-time augmentation + calibration (T=0.95)", f"{CLASSES[int(probs.argmax())]} {probs.max():.0%}")
    agree = int(p1.argmax()) == int(p2.argmax())
    if not agree or probs.max() < UNSURE_CUTOFF:
        act("decide", "networks disagree or confidence < 70%: escalate to a second look")
        views = [T.Compose([T.Resize((s_, s_)), T.CenterCrop(IMG_SIZE), T.ToTensor()]) for s_ in (232, 256, 272)]
        extra = [probs_of(m, pil, tf) for m in (model, model2) for tf in views]
        p_second = np.mean(extra, 0); same = int(p_second.argmax()) == int(probs.argmax())
        act("second look: 6 extra zoomed views", f"{CLASSES[int(p_second.argmax())]} {p_second.max():.0%} ({'confirms' if same else 'CONTRADICTS'} first read)")
        probs = (probs * 4 + p_second * 6) / 10
        act("update belief", f"{CLASSES[int(probs.argmax())]} {probs.max():.0%}")
    else:
        act("decide", "networks agree and confident: no second look needed")
    adj = probs.copy(); adj[1] += FINAL_BONUS; cls = int(adj.argmax())
    act("apply safety rule (+0.58 to bacterial)", f"decision: {CLASSES[cls]}")
    conf = float(probs.max()); ent = float(-(probs * np.log(probs + 1e-9)).sum() / np.log(K))
    cam = gradcam(pil, model2, cls)
    base = np.array(pil.convert("RGB").resize((IMG_SIZE, IMG_SIZE))) / 255.0
    overlay = ((0.65 * base + 0.35 * plt.cm.jet(cam)[..., :3]) * 255).astype(np.uint8)
    lung = float(cam[int(.12*IMG_SIZE):int(.92*IMG_SIZE), int(.15*IMG_SIZE):int(.85*IMG_SIZE)].sum() / (cam.sum() + 1e-8))
    act("Grad-CAM: where did it look?", f"{lung:.0%} of attention inside the lungs" + ("" if lung >= 0.5 else ": FLAG possible shortcut"))
    every = None; fed_child = fed_adult = "n/a"
    if FED_OK:
        fp = probs_of(fed_model, pil); every = {"normal": float(fp[0]), "pneumonia": float(fp[1])}
        fed_child = "pneumonia" if fp[1] >= CHILD_TH else "normal"; fed_adult = "pneumonia" if fp[1] >= ADULT_TH else "normal"
        act("consult federated all-ages model", f"pneumonia {fp[1]:.0%}" + (" (agrees)" if (fp[1] >= 0.5) == (cls != 0) else " (DISAGREES: flag for review)"))
    cset = [c for c, p in zip(CLASSES, probs) if p >= CONF_MIN] or [CLASSES[int(probs.argmax())]]
    act("conformal 95% guarantee", " / ".join(cset))
    SET_TEXT = {("normal",): "Normal", ("bacterial",): "Bacterial pneumonia", ("viral",): "Viral pneumonia",
                ("bacterial", "viral"): "Pneumonia, type unclear (bacterial or viral)",
                ("normal", "bacterial", "viral"): "Can't narrow it down: refer"}
    set_txt = SET_TEXT.get(tuple(cset), " or ".join(cset))
    if conf < UNSURE_CUTOFF or not agree:
        why = "the two networks disagree" if not agree else f"confidence {conf:.0%} is below {UNSURE_CUTOFF:.0%}"
        head = f"## UNSURE: refer to a doctor / radiologist\n*Why:* {why}."
    else:
        head = {0: "## Normal", 1: "## Bacterial pneumonia: likely needs antibiotics, see a doctor now", 2: "## Viral pneumonia: see a doctor"}[cls]
    head += (f"\n\n**95% guaranteed answer (conformal prediction):** {set_txt}  \n"
             f"*The true diagnosis is inside this answer 95% of the time (96.4% on 799 unseen X-rays). "
             f"One label = confident; more labels = the model honestly can't tell.*")
    level = max([c for c in TRACK if c <= conf], default=None)
    if level is not None:
        head += f"\n\nOn held-out test X-rays where PneumoType was at least {level:.0%} confident, it was right **{TRACK[level]:.0%}** of the time."
    head += "\n\n*Prototype only. Not medical advice.*"
    stats = pd.DataFrame([
        ["CNN 1 says", f"{CLASSES[int(p1.argmax())]} ({p1.max():.0%})"],
        ["CNN 2 (clinic-robust) says", f"{CLASSES[int(p2.argmax())]} ({p2.max():.0%})"],
        ["Networks agree?", "yes" if agree else "NO: flagged as unsure"],
        ["Ensemble + TTA (calibrated) top class", f"{CLASSES[int(probs.argmax())]} ({conf:.0%})"],
        ["Conformal 95% set", " / ".join(cset) + f"  (labels with prob >= {CONF_MIN:.2f})"],
        ["Decision after safety tuning", CLASSES[cls]],
        ["Uncertainty (normalized entropy)", f"{ent:.2f}  (0 = certain, 1 = coin flip)"],
        ["Safety bonus applied to bacterial", f"+{FINAL_BONUS:.2f}"],
        ["Heatmap focus inside lung region", f"{lung:.0%}"],
        ["Pneumonia check, anyone (federated)", f"{every['pneumonia']:.0%}" if every else "n/a"],
        ["Federated verdict at children's / adult hospital threshold", f"{fed_child} (>= {CHILD_TH}) / {fed_adult} (>= {ADULT_TH})" if every else "n/a"],
        ["Analysis time", f"{(time.time() - t0) * 1000:.0f} ms"],
    ], columns=["Signal", "Value"])
    unsure = conf < UNSURE_CUTOFF or not agree
    act("input safety checks + write triage report", "done")
    report, rpath = make_report(cls, cset, set_txt, probs, conf, agree, unsure, input_checks(pil, lung), every, fed_child, fed_adult, (time.time() - t0) * 1000)
    agent_md = "### Agent log: what PneumoType decided to do\n| time | action | result |\n|---|---|---|\n" + "\n".join(f"| {a} | {b} | {c} |" for a, b, c in log)
    return head, {c: float(p) for c, p in zip(CLASSES, probs)}, every, overlay, stats, agent_md + "\n\n" + report, rpath

In [ ]:
import gradio as gr
# example X-rays from the children's test set, if the dataset is attached
ex = sorted(glob.glob("/kaggle/input/**/test/PNEUMONIA/*.jpeg", recursive=True))[:4] + sorted(glob.glob("/kaggle/input/**/test/NORMAL/*.jpeg", recursive=True))[:2]
def fetch(url):
    try: return np.array(Image.open(io.BytesIO(urllib.request.urlopen(url, timeout=20).read())).convert("RGB"))
    except Exception: return None
REPO = "https://raw.githubusercontent.com/Riyadua777/pneumotype/main/figures/"
SYSTEM_IMG, RESULTS_IMG = fetch(REPO + "system.png"), fetch(REPO + "results.png")

METRICS = '''### Children: normal / bacterial / viral (held-out test, 799 X-rays)
| | Simple baseline | **PneumoType** | **+ safety tuning** |
|---|---|---|---|
| Accuracy | 77.1% | **84.4%** | 81.6% |
| Macro recall | 77.2% | **84.2%** | 77.5% |
| Pneumonia caught | 95.3% | 97.2% | **99.3%** |
| Dangerous misses (bacterial) | 92 | 62 | **18** |
| False alarms (of 222 normal) | 25 | **7** | 18 |

**95% confidence intervals (bootstrapped by patient):** accuracy 84.4% [81.6–87.0], pneumonia caught 97.2% [95.8–98.5]
**Conformal prediction (95% target):** true diagnosis inside the answer **96.4%** of the time; single-label answers right **95.0%** of the time
**Calibration error (ECE):** 0.044 → 0.037 after temperature scaling · **When ≥90% confident:** right 97.7% of the time
**Robustness:** first CNN fell to 0.66 on low-contrast X-rays; the clinic-robust CNN stays 0.83 on every kind of image
**Shortcut test:** lungs blacked out → chance (0.34), so it reads the lungs, not the borders
**Federated learning (children + adults, no X-rays shared):** balanced accuracy 88.4% on children, 83.1% on adults (best worst-case of any private setup); a children's-hospital threshold raises pneumonia caught from 82% to 87%'''

## 0. Built for clinics with very little
No radiologist, no big computer, maybe no internet. Our CNNs are tiny, so the diagnosis itself runs **offline on a cheap laptop CPU**. We measure that here.

In [ ]:
import copy, os
size_mb = sum(os.path.getsize(find(n)) for n in ["best.pt", "best_robust.pt"]) / 1e6
cpu_m1, cpu_m2 = copy.deepcopy(model).cpu().eval(), copy.deepcopy(model2).cpu().eval()
x = torch.rand(1, 1, IMG_SIZE, IMG_SIZE)
torch.set_num_threads(2)   # like a cheap 2-core laptop
with torch.no_grad():
    for _ in range(2): cpu_m1(x)
    t = time.time()
    for _ in range(10): cpu_m1(x); cpu_m2(x)
cpu_ms = (time.time() - t) / 10 * 1000
torch.set_num_threads(os.cpu_count())
FOOTPRINT = f"Both CNNs together: {size_mb:.0f} MB on disk, {cpu_ms:.0f} ms per X-ray on 2 CPU cores, no GPU, no internet."
print(FOOTPRINT); del cpu_m1, cpu_m2

## 1. The language model (coordinator only)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import json, re
LLM_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # small enough for an ordinary laptop
tok = AutoTokenizer.from_pretrained(LLM_ID)
llm = AutoModelForCausalLM.from_pretrained(LLM_ID, dtype=torch.float16).to(device).eval()

@torch.no_grad()
def chat(messages, max_new_tokens=220):
    enc = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    ids, mask = enc["input_ids"].to(device), enc["attention_mask"].to(device)
    out = llm.generate(input_ids=ids, attention_mask=mask, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

print(chat([{"role": "user", "content": "Reply with the single word: ready"}]))

## 2. The tools the agent can call
Each tool is one of our own components. The agent sees only the tool's JSON result.

In [ ]:
TRIAGE_LEVELS = ["ROUTINE", "REVIEW", "REVIEW TODAY", "URGENT", "REFER"]   # REFER = can't triage, human must read
TOOL_DOCS = {
    "check_image_quality": "Checks the upload is a usable greyscale chest X-ray (colour, resolution, contrast).",
    "classify_xray": "Runs our two CNNs (trained from scratch) plus a calibrated ensemble. Returns probabilities for normal / bacterial / viral and whether the two networks agree.",
    "second_look": "Re-examines the X-ray at 3 extra zoom levels with both CNNs. Use when the networks disagree or the top probability is below 0.70.",
    "attention_check": "Grad-CAM: share of the model's attention inside the lungs. Below 0.5 means a possible shortcut.",
    "conformal_set": "The set of diagnoses guaranteed to contain the true one 95% of the time (validated: 96.4% on 799 unseen X-rays). One label = confident; two or more = genuinely uncertain.",
    "federated_check": "Pneumonia vs normal from a model trained by federated learning across a children's and an adult hospital (works for all ages).",
}
REQUIRED = ["check_image_quality", "classify_xray", "attention_check", "conformal_set"]

class Case:
    # lazily computes tool results for one X-ray
    def __init__(self, pil): self.pil = pil; self.r = {}
    def probs(self):
        if "probs" not in self.r:
            p1, p2 = probs_of(model, self.pil), probs_of(model2, self.pil); p = final_probs(self.pil)
            self.r.update(p1=p1, p2=p2, probs=p)
        return self.r["probs"]
    def run(self, name):
        if name == "check_image_quality":
            lung = self.lung()
            checks = input_checks(self.pil, lung)[:3]
            return {"checks": [f"{a}: {b}" for a, b in checks], "usable": all(a != "FAIL" for a, b in checks)}
        if name == "classify_xray":
            p = self.probs(); p1, p2 = self.r["p1"], self.r["p2"]
            return {"probabilities": {c: round(float(v), 3) for c, v in zip(CLASSES, p)},
                    "cnn1_says": CLASSES[int(p1.argmax())], "cnn2_says": CLASSES[int(p2.argmax())],
                    "networks_agree": bool(p1.argmax() == p2.argmax())}
        if name == "second_look":
            views = [T.Compose([T.Resize((s, s)), T.CenterCrop(IMG_SIZE), T.ToTensor()]) for s in (232, 256, 272)]
            ps = np.mean([probs_of(m, self.pil, tf) for m in (model, model2) for tf in views], 0)
            first = self.probs(); self.r["probs"] = (first * 4 + ps * 6) / 10
            return {"second_look_probabilities": {c: round(float(v), 3) for c, v in zip(CLASSES, ps)},
                    "confirms_first_read": bool(ps.argmax() == first.argmax()),
                    "updated_probabilities": {c: round(float(v), 3) for c, v in zip(CLASSES, self.r["probs"])}}
        if name == "attention_check":
            l = self.lung(); return {"share_of_attention_in_lungs": round(l, 2), "possible_shortcut": l < 0.5}
        if name == "conformal_set":
            p = self.probs(); s = [c for c, v in zip(CLASSES, p) if v >= CONF_MIN] or [CLASSES[int(p.argmax())]]
            self.r["cset"] = s; return {"guaranteed_95_set": s}
        if name == "federated_check":
            if not FED_OK: return {"error": "federated model not loaded"}
            fp = probs_of(fed_model, self.pil); self.r["fed"] = float(fp[1])
            return {"pneumonia_probability": round(float(fp[1]), 3), "verdict": "pneumonia" if fp[1] >= 0.5 else "normal"}
        return {"error": f"unknown tool {name}"}
    def lung(self):
        if "lung" not in self.r:
            p = self.probs(); adj = p.copy(); adj[1] += FINAL_BONUS
            cam = gradcam(self.pil, model2, int(adj.argmax()))
            self.r["cam"] = cam
            self.r["lung"] = float(cam[int(.12*IMG_SIZE):int(.92*IMG_SIZE), int(.15*IMG_SIZE):int(.85*IMG_SIZE)].sum() / (cam.sum() + 1e-8))
        return self.r["lung"]

def policy_triage(case):
    # fixed safety policy, computed from tool results only (same rules as the main demo)
    p = case.probs(); adj = p.copy(); adj[1] += FINAL_BONUS; cls = int(adj.argmax())
    s = case.r.get("cset") or [c for c, v in zip(CLASSES, p) if v >= CONF_MIN]
    usable = all(a != "FAIL" for a, b in input_checks(case.pil, case.lung())[:1])
    unsure = p.max() < UNSURE_CUTOFF or not (case.r["p1"].argmax() == case.r["p2"].argmax())
    if not usable or unsure or len(s) == 3: return "REFER"
    if cls == 1 and "bacterial" in s: return "URGENT"
    if "normal" not in s: return "REVIEW TODAY" if len(s) == 2 else "REVIEW"
    if s == ["normal"]: return "ROUTINE"
    return "REVIEW"
print("tools ready:", list(TOOL_DOCS))

## 3. The agent loop
The language model replies with JSON: either a tool call or a final triage. We parse it, run the tool, and feed back the result. **Guardrails:** (1) it can't finish before calling every required safety tool, (2) a triage less urgent than the safety policy is overridden, (3) invalid JSON gets one retry, then the next required tool is run automatically.

In [ ]:
SYSTEM = ("You are PneumoType's triage agent at a children's clinic without a radiologist. You cannot see the X-ray; "
          "you can only call tools, one per turn. Never invent numbers. Tools:\n" +
          "\n".join(f"- {k}: {v}" for k, v in TOOL_DOCS.items()) +
          "\nRules: start with check_image_quality, then classify_xray. If the networks disagree or the top probability is "
          "below 0.70, call second_look. Always call attention_check and conformal_set before finishing. "
          "federated_check is optional (useful for a second opinion).\n"
          "Reply with ONLY one JSON object, either\n"
          '{"thought": "<short reasoning>", "tool": "<tool name>"}\n'
          "or, when done,\n"
          '{"thought": "<short reasoning>", "final": {"triage": "<URGENT | REVIEW TODAY | REVIEW | ROUTINE | REFER>", '
          '"summary": "<2-3 plain sentences for the clinician>"}}')

def parse(txt):
    m = re.search(r"\{.*\}", txt, re.S)
    if not m: return None
    try: return json.loads(m.group(0))
    except Exception:
        try: return json.loads(m.group(0).replace("'", '"'))
        except Exception: return None

def run_agent(pil, max_steps=9):
    case = Case(pil); msgs = [{"role": "system", "content": SYSTEM},
                              {"role": "user", "content": "A new chest X-ray has been uploaded. Triage it."}]
    trace, called, stats = [], [], {"invalid_json": 0, "enforced": 0, "voluntary_required": 0}
    final = None; t0 = time.time()
    for step in range(max_steps):
        out = chat(msgs); j = parse(out)
        if j is None:
            stats["invalid_json"] += 1
            missing = [t for t in REQUIRED if t not in called]
            j = {"thought": "(agent reply was not valid JSON: guardrail runs the next required tool)", "tool": missing[0] if missing else None}
            if j["tool"] is None: break
        if "final" in j:
            missing = [t for t in REQUIRED if t not in called]
            if missing:
                stats["enforced"] += 1
                trace.append(("guardrail", f"tried to finish without {', '.join(missing)}: running it", ""))
                j = {"thought": "guardrail", "tool": missing[0]}
            else:
                final = j["final"]; trace.append(("final", j.get("thought", ""), "")); break
        tool = j.get("tool")
        if tool not in TOOL_DOCS:
            obs = {"error": f"unknown tool '{tool}'. Valid tools: {list(TOOL_DOCS)}"}
        else:
            obs = case.run(tool); called.append(tool)
        trace.append((tool, j.get("thought", ""), json.dumps(obs)))
        msgs += [{"role": "assistant", "content": out if parse(out) else json.dumps(j)},
                 {"role": "user", "content": f"Observation from {tool}: {json.dumps(obs)}"}]
    for t in REQUIRED:   # make sure the safety tools ran even if the step budget ran out
        if t not in called: case.run(t); called.append(t); stats["enforced"] += 1; trace.append(("guardrail", f"step budget used up: ran {t}", ""))
    stats["voluntary_required"] = sum(1 for t in REQUIRED if t in [x[0] for x in trace if x[1] != "guardrail"])
    pol = policy_triage(case)
    agent_triage = (final or {}).get("triage", "").upper().strip()
    if agent_triage not in TRIAGE_LEVELS: agent_triage = None
    override = agent_triage is None or TRIAGE_LEVELS.index(agent_triage) < TRIAGE_LEVELS.index(pol)
    triage = pol if override else agent_triage
    summary = (final or {}).get("summary", "") or "(no summary)"
    return dict(case=case, trace=trace, called=called, agent_triage=agent_triage, policy=pol, triage=triage,
                override=override, summary=summary, stats=stats, seconds=time.time() - t0, steps=len(trace))

def render(res):
    rows = "\n".join(f"| {i+1} | `{t}` | {th} | {('`' + o[:160] + ('…' if len(o) > 160 else '') + '`') if o else ''} |"
                     for i, (t, th, o) in enumerate(res["trace"]))
    g = (f"⚠️ **Guardrail override:** the agent said **{res['agent_triage'] or 'nothing valid'}**, but the safety policy requires "
         f"**{res['policy']}**, so the policy wins.") if res["override"] else \
        f"✅ The agent's triage (**{res['agent_triage']}**) passes the safety policy (minimum: {res['policy']})."
    return (f"## Triage: **{res['triage']}**\n{res['summary']}\n\n{g}\n\n"
            f"*{res['steps']} steps · {res['seconds']:.1f} s · tools called: {', '.join(res['called'])} · invalid replies: {res['stats']['invalid_json']}*\n\n"
            f"### Agent trace: what it decided to do\n| # | action | agent's reasoning | tool result |\n|---|---|---|---|\n{rows}\n\n"
            "*Prototype decision support, not medical advice. The language model coordinates; our CNNs and the safety policy decide.*")

LANGS = ["English", "Spanish", "French", "Portuguese", "Hindi", "Bengali", "Swahili"]
def translate(text, lang):
    if lang == "English" or not text: return text
    return chat([{"role": "system", "content": f"Translate the user's text into simple, plain {lang} for a community health worker. "
                  "Keep the triage word (URGENT, REVIEW TODAY, REVIEW, ROUTINE, REFER) in English. Output only the translation."},
                 {"role": "user", "content": text}], max_new_tokens=260)

POLICY_WHY = {
    "URGENT": "bacterial pneumonia is possible and is the most likely type; missing bacterial pneumonia is the most dangerous error, so the safety policy treats it as urgent",
    "REVIEW TODAY": "pneumonia is very likely but bacterial vs viral cannot be told apart from the X-ray, so a clinician should review today",
    "REVIEW": "a viral pattern or a mixed picture; a clinician should review",
    "ROUTINE": "normal is the only diagnosis in the 95% guaranteed set",
    "REFER": "the system is not confident or the image is not usable, so a human must read it",
}
def facts(res):
    c = res["case"]; p = c.probs(); s = c.r.get("cset", [])
    lines = [f"Most likely: {CLASSES[int(p.argmax())]} ({p.max():.0%}). Probabilities: bacterial {p[1]:.0%}, viral {p[2]:.0%}, normal {p[0]:.0%}.",
             f"The X-ray is {'NOT ' if 'normal' not in s else ''}normal according to the 95% guaranteed set: {' or '.join(s)}.",
             f"Both networks agree: {'yes' if c.r['p1'].argmax() == c.r['p2'].argmax() else 'no'}."]
    if "lung" in c.r: lines.append(f"{c.r['lung']:.0%} of the model's attention is inside the lungs ({'possible shortcut' if c.r['lung'] < 0.5 else 'no shortcut flag'}).")
    if "fed" in c.r: lines.append(f"Federated all-ages model: pneumonia probability {c.r['fed']:.0%}.")
    lines.append(f"Final triage: {res['triage']}, because {POLICY_WHY[res['triage']]}.")
    if res["override"]: lines.append(f"The language agent first suggested {res['agent_triage']}, but the safety policy raised it to {res['triage']}.")
    return " ".join(lines)

def ask(question, res, lang="English"):
    if not res: return "Run the agent on an X-ray first."
    msgs = [{"role": "system", "content": "You are PneumoType's triage assistant for a community health worker. Answer in at most 3 short sentences, "
             f"in simple {lang}, using ONLY these facts. Do not add new findings. Never give drug names or doses. Facts: {facts(res)}"},
            {"role": "user", "content": question}]
    return chat(msgs, max_new_tokens=160)
print("agent ready")

## 4. Evaluating the agent
We run the agent on X-rays of every type and measure **how it behaves**, not just what it says: did it call the safety tools by itself, how often did a guardrail have to step in, and how long did it take.
*Accuracy is measured in the main notebook. These example X-rays come from the dataset's original test folder, which may overlap with our training split, so here we only report agent behaviour.*

In [ ]:
pool = {"normal": sorted(glob.glob("/kaggle/input/**/test/NORMAL/*.jpeg", recursive=True)),
        "bacterial": sorted(f for f in glob.glob("/kaggle/input/**/test/PNEUMONIA/*.jpeg", recursive=True) if "bacteria" in f),
        "viral": sorted(f for f in glob.glob("/kaggle/input/**/test/PNEUMONIA/*.jpeg", recursive=True) if "virus" in f)}
N_EACH = 6
rows = []
for lab, files in pool.items():
    for f in files[:N_EACH]:
        r = run_agent(Image.open(f).convert("RGB"))
        rows.append({"true label": lab, "agent triage": r["agent_triage"], "policy minimum": r["policy"], "final": r["triage"],
                     "override": r["override"], "steps": r["steps"], "invalid JSON": r["stats"]["invalid_json"],
                     "safety tools enforced": r["stats"]["enforced"], "second look used": "second_look" in r["called"],
                     "seconds": round(r["seconds"], 1)})
agent_eval = pd.DataFrame(rows)
summary = pd.Series({
    "X-rays": len(agent_eval),
    "ran every safety tool without being forced": f"{(agent_eval['safety tools enforced'] == 0).mean():.0%}",
    "replies that were valid JSON": f"{1 - agent_eval['invalid JSON'].sum() / agent_eval['steps'].sum():.0%}",
    "guardrail overrides (agent too relaxed)": f"{agent_eval['override'].mean():.0%}",
    "used second look": f"{agent_eval['second look used'].mean():.0%}",
    "average steps": f"{agent_eval['steps'].mean():.1f}",
    "average seconds per X-ray": f"{agent_eval['seconds'].mean():.1f}",
})
print("=== AGENT EVALUATION ==="); print(summary.to_string()); print(); print(agent_eval.to_string())

## 5. The app

In [ ]:
def agent_ui(img, lang):
    if img is None: return "### Upload or pick an X-ray", None, None
    pil = Image.fromarray(img) if not isinstance(img, Image.Image) else img
    res = run_agent(pil)
    if lang != "English": res["summary"] = translate(res["summary"], lang) + f"\n\n*(English: {res['summary']})*"
    cam = res["case"].r.get("cam")
    overlay = None
    if cam is not None:
        base = np.array(pil.convert("RGB").resize((IMG_SIZE, IMG_SIZE))) / 255.0
        overlay = ((0.65 * base + 0.35 * plt.cm.jet(cam)[..., :3]) * 255).astype(np.uint8)
    return render(res), overlay, res

def ask_ui(q, history, res, lang):
    history = (history or []) + [{"role": "user", "content": q}, {"role": "assistant", "content": ask(q, res, lang)}]
    return history, ""

with gr.Blocks(title="PneumoType Agent") as demo:
    gr.Markdown("# PneumoType Agent: AI triage for chest X-rays\nA language model **coordinates** our own CNNs (trained from scratch), "
                "a conformal 95% guarantee, a Grad-CAM shortcut check and a federated all-ages model, under fixed **safety guardrails**. "
                "Built for clinics with **no radiologist**: the diagnosis runs offline on a cheap laptop, and reports come in the health worker's language. "
                "*Prototype, not medical advice.*  \n**Footprint:** " + FOOTPRINT)
    state = gr.State()
    with gr.Tab("Agent"):
        with gr.Row():
            with gr.Column(scale=1):
                a_inp = gr.Image(label="Chest X-ray", type="numpy")
                lang = gr.Dropdown(LANGS, value="English", label="Report language (for community health workers)")
                a_btn = gr.Button("Run triage agent", variant="primary")
                if ex: gr.Examples(ex, inputs=a_inp, label="Example X-rays (click one, then Run)")
                a_cam = gr.Image(label="Grad-CAM: where the CNN looked")
            with gr.Column(scale=2):
                a_out = gr.Markdown()
                gr.Markdown("### Ask the agent")
                chatbox = gr.Chatbot(height=260)
                q = gr.Textbox(placeholder="e.g. Why is this urgent? Should I worry the model is guessing?", label="Your question")
        a_btn.click(agent_ui, [a_inp, lang], [a_out, a_cam, state])
        q.submit(ask_ui, [q, chatbox, state, lang], [chatbox, q])
    with gr.Tab("Agent evaluation"):
        gr.Markdown("### How the agent behaved on %d X-rays\n" % len(agent_eval) + "| measure | value |\n|---|---|\n" + "\n".join(f"| {k} | {v} |" for k, v in summary.items()))
        gr.Dataframe(agent_eval)
    with gr.Tab("Quick diagnose (no agent)"):
        with gr.Row():
            with gr.Column(scale=1):
                inp = gr.Image(label="Chest X-ray", type="numpy")
                if ex: gr.Examples(ex, inputs=inp, label="Example X-rays")
            with gr.Column(scale=2):
                verdict = gr.Markdown()
                with gr.Row():
                    type_lbl = gr.Label(label="Pneumonia type (calibrated)", num_top_classes=3)
                    every_lbl = gr.Label(label="Pneumonia check for anyone (federated)")
                with gr.Row():
                    cam_out = gr.Image(label="Grad-CAM")
                    stats_out = gr.Dataframe(label="Under the hood", wrap=True)
                report_out = gr.Markdown(); report_file = gr.File(label="Download triage report")
        inp.change(analyze, inp, [verdict, type_lbl, every_lbl, cam_out, stats_out, report_out, report_file])
    with gr.Tab("How good is it?"):
        gr.Markdown(METRICS)
        if RESULTS_IMG is not None: gr.Image(RESULTS_IMG, label="Results dashboard (held-out test data)")
    with gr.Tab("How it works"):
        if SYSTEM_IMG is not None: gr.Image(SYSTEM_IMG, label="The PneumoType system")
        gr.Markdown('''1. **Our models** (trained from scratch): two CNNs, an ensemble with calibration, and a federated model across a children's and an adult hospital.
2. **The agent**: Qwen2.5-1.5B-Instruct (small open model, not ours) only decides **which of our tools to call** and explains results. It never sees pixels.
3. **Guardrails**: required safety tools are enforced; a fixed safety policy overrides any triage that is less urgent than the evidence allows.
4. **Conformal prediction**: a 95%-guaranteed set of diagnoses (96.4% on 799 unseen X-rays).
5. **Shortcut checks**: Grad-CAM attention inside the lungs; lungs-blacked-out tests in the main notebook.''')
demo.launch(share=True)

In [ ]:
# keep the session busy while you record
while True: time.sleep(60)